# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 6: Introduction to Machine Learning

**Instructor:** Dr. Eyuphan Koc
**Department of Civil Engineering, Bogazici University**
**Semester:** Spring 2026

---

## Table of Contents

1. [What is Machine Learning?](#1.-What-is-Machine-Learning?)
2. [The Scikit-Learn Workflow](#2.-The-Scikit-Learn-Workflow)
3. [The Accuracy Trap](#3.-The-Accuracy-Trap)
4. [The Confusion Matrix](#4.-The-Confusion-Matrix)
5. [Precision, Recall, and the Tradeoff](#5.-Precision,-Recall,-and-the-Tradeoff)
6. [ROC Curves and AUC](#6.-ROC-Curves-and-AUC)
7. [Overfitting](#7.-Overfitting)
8. [Data Leakage](#8.-Data-Leakage)
9. [Putting It Together](#9.-Putting-It-Together)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve, auc,
                             mean_squared_error, r2_score)
%matplotlib inline

We begin by importing the libraries we will use throughout this lecture. Don't worry if you don't recognize all of these yet — we will introduce each one as we need it.

---

## 1. What is Machine Learning?

After the devastating 1999 Izmit earthquake, engineers had borehole data from thousands of sites across the Marmara region. They wanted to answer a critical question: *which sites are at risk of soil liquefaction in a future quake?*

They could try writing explicit rules — *"if SPT blow count is below 15 and the groundwater table is shallow, flag it"* — but how do you set those thresholds? What about the interactions between soil type, depth, seismic intensity, and dozens of other factors?

**Machine learning takes a different approach:** instead of programming rules, you provide examples and let the algorithm discover the patterns.

> **Definition: Machine Learning**
>
> Machine Learning is about building **mathematical models** to understand data. Models learn from examples rather than explicit programming.
>
> *"Instead of coding rules for predicting soil behavior, we provide examples of sites that did and didn't liquefy — the model learns the relationship."*

### Categories of Machine Learning

| | **Supervised Learning** | **Unsupervised Learning** |
|---|---|---|
| **Data** | Learn from **labeled** data | Learn from **unlabeled** data |
| **Setup** | Have input-output pairs | No predefined outputs |
| **Goal** | Predict outputs for new inputs | Discover structure |
| **Type 1** | **Classification**: Predict discrete labels | **Clustering**: Group similar items |
| **Type 2** | **Regression**: Predict continuous values | **Dimensionality Reduction**: Compress data |

> **Key Insight: The Key Difference**
> Supervised: *"Here are examples with answers — learn the pattern"* vs Unsupervised: *"Find patterns in this data on your own"*

### Supervised Learning: Classification vs Regression

| | **Classification** | **Regression** |
|---|---|---|
| **Predict** | **Discrete categories** | **Continuous values** |
| **Output** | A label or class | A number |
| **CE Examples** | Soil: will it liquefy in an earthquake? | Building heating load from geometry |
| | Pipe segment: will it fail this year? | Traffic speed from vehicle density |

### Visualizing Classification and Regression

The following plot shows the key difference between classification (left) and regression (right).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Classification: Liquefied vs Stable soil samples
ax = axes[0]
# Stable sites (high SPT blow count, deeper)
ax.scatter([25, 30, 35, 28, 32], [3, 8, 12, 5, 10],
           c='steelblue', s=80, label='Stable', edgecolors='k', zorder=3)
# Liquefied sites (low SPT blow count, shallower)
ax.scatter([5, 8, 12, 7], [2, 4, 3, 6],
           c='indianred', s=80, label='Liquefied', edgecolors='k', zorder=3)
ax.set_xlabel('SPT Blow Count (N)', fontsize=12)
ax.set_ylabel('Depth (m)', fontsize=12)
ax.set_title('Classification: Liquefaction Risk', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Regression: Wall area vs heating load
ax = axes[1]
wall_pts = np.array([250, 280, 320, 360, 400])
heat_pts = np.array([13, 17, 23, 30, 38])
ax.scatter(wall_pts, heat_pts, c='indianred', s=80, edgecolors='k', zorder=3, label='Buildings')
x_line = np.linspace(240, 420, 100)
y_line = -19 + 0.14 * x_line
ax.plot(x_line, y_line, 'steelblue', linewidth=2, label='Fit line')
ax.set_xlabel('Wall Area (m²)', fontsize=12)
ax.set_ylabel('Heating Load (kWh/m²)', fontsize=12)
ax.set_title('Regression: Energy Efficiency', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Machine Learning Workflow

Every ML project follows the same high-level pipeline:

```
 [Raw Data] --> [Preprocess & Clean] --> [Feature Engineering]
                                                  |
                                                  v
                    [Validate] <-- [Train Model] <-- [Choose Model]
                        |                ^
                        |                |  (adjust)
                        +----------------+
                        |
                        v
                     [Deploy]
```

**Key Steps:**
1. **Data Collection & Preprocessing:** Gather, clean, normalize data
2. **Feature Engineering:** Select/create informative features
3. **Model Selection & Training:** Choose algorithm, fit to data
4. **Validation:** Test on unseen data, tune hyperparameters
5. **Deployment:** Use model in production

> **[QUICK]** For each scenario below, decide: *supervised or unsupervised? Classification or regression?*
>
> 1. Predicting a building's annual heating load from its wall area, roof area, and glazing percentage
> 2. Grouping soil samples into clusters based on grain size, plasticity, and moisture (no labels)
> 3. Deciding whether a soil layer will liquefy during an earthquake based on borehole data
> 4. Estimating traffic speed on a highway from vehicle density measurements

---

## 2. The Scikit-Learn Workflow

Scikit-learn is Python's premier machine learning library. Its most powerful feature is a **consistent API** — learning one model teaches you all models.

> **Definition: The Estimator API**
>
> All scikit-learn models follow the same three-step pattern:
>
> ```python
> from sklearn.some_module import SomeModel
>
> model = SomeModel(hyperparameters)  # 1. Choose model
> model.fit(X, y)                     # 2. Fit to data
> predictions = model.predict(X_new)  # 3. Apply to new data
> ```

### Data Representation in Scikit-Learn

> **Definition: Features Matrix and Target Array**
>
> | | **Features Matrix: $\mathbf{X}$** | **Target Array: $\mathbf{y}$** |
> |---|---|---|
> | **Shape** | $[n_{\text{samples}}, n_{\text{features}}]$ | $[n_{\text{samples}}]$ |
> | **Content** | Each row = one sample, each column = one feature | Labels (classification) or values (regression) |
> | **Type** | 2D NumPy array or pandas DataFrame | 1D NumPy array or pandas Series |

### [TOGETHER] Example: Predicting Building Heating Load

**Problem:** Predict a building's heating load (kWh/m²) from its wall area.

This is inspired by the [UCI Energy Efficiency Dataset](https://archive.ics.uci.edu/dataset/242/energy+efficiency) — a real dataset of 768 buildings analyzed for energy performance. Let's see the Estimator API in action.

In [ ]:
# Building wall area vs heating load (inspired by UCI Energy Efficiency dataset)
X_wall = np.array([[245], [280], [318], [350], [390], [416]])
y_heat = np.array([12, 16, 22, 28, 35, 41])  # Heating load in kWh/m²

# 1. Choose model
model_energy = LinearRegression()

# 2. Fit model
model_energy.fit(X_wall, y_heat)

# 3. Make predictions
X_new_wall = np.array([[300], [375]])
predictions_energy = model_energy.predict(X_new_wall)

print(f"Predictions: {predictions_energy.round(1)} kWh/m²")
print(f"Slope: {model_energy.coef_[0]:.4f} kWh/m² per m² of wall area")
print(f"Intercept: {model_energy.intercept_:.2f}")

In [ ]:
# Visualize the regression fit
X_line_wall = np.linspace(230, 430, 100).reshape(-1, 1)
y_line_wall = model_energy.predict(X_line_wall)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_wall, y_heat, color='steelblue', s=100, alpha=0.7,
           label='Training Data', edgecolors='k', zorder=3)
ax.plot(X_line_wall, y_line_wall, color='indianred', linewidth=2, label='Regression Line')
ax.scatter(X_new_wall, predictions_energy, color='green', s=120, marker='s',
           label='Predictions', zorder=5, edgecolors='k')
ax.set_xlabel('Wall Area (m²)', fontsize=12)
ax.set_ylabel('Heating Load (kWh/m²)', fontsize=12)
ax.set_title('Linear Regression: Wall Area vs Building Heating Load', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Key Insight: The Estimator API is Universal**
>
> Whether you're doing linear regression, logistic regression, decision trees, or neural networks — the pattern is always `fit()` → `predict()` → `score()`. Master this pattern once, and you can use any model in scikit-learn.

> **[QUICK]** What would you change to predict heating load from *multiple* building features (wall area, roof area, glazing percentage, overall height)?
> *Hint: Only the shape of X needs to change — the API stays the same.*

---

## 3. The Accuracy Trap

Now that you know how to train a model, the natural question is: *how good is it?*

Every beginner's first question about a model is: *"What's the accuracy?"*

This section shows why that question, on its own, can be dangerously misleading — especially in geotechnical engineering, where soil failures are rare but catastrophic.

> **Example: Everyday Accuracy Traps**
>
> Before we look at any code, consider these scenarios:
>
> - A weather model that always predicts "no rain" in Izmir would be right ~90% of the time. But you'd never trust it during the rainy season.
> - A spam filter that never flags anything as spam has 100% accuracy on legitimate emails — but lets every phishing attack through.
> - A medical screening test that always says "healthy" would have >99% accuracy, because most people ARE healthy. But it would miss every case of disease.
>
> **The pattern:** when one outcome is much rarer than the other, a lazy model that always predicts the common outcome gets high accuracy *for free*.

In [ ]:
# (continuing with the same imports from above)

In [ ]:
np.random.seed(42)

# Soil liquefaction dataset — inspired by SPT-based records from the 1999 Izmit earthquake
# HIGHLY IMBALANCED: most sites remain stable
n_stable = 900
n_liquefied = 100

# Features for stable sites (higher SPT N, deeper groundwater, lower PGA)
X_stable = np.column_stack([
    np.random.uniform(8, 45, n_stable),      # spt_n: SPT blow count
    np.random.uniform(1, 20, n_stable),      # depth (m)
    np.random.uniform(0, 70, n_stable),      # fines_content (%)
    np.random.uniform(0.05, 0.45, n_stable), # pga: peak ground acceleration (g)
    np.random.uniform(1, 10, n_stable)       # gw_depth: groundwater depth (m)
])

# Features for liquefied sites (low SPT N, shallow GW, high PGA, high fines)
X_liquefied = np.column_stack([
    np.random.uniform(2, 25, n_liquefied),
    np.random.uniform(1, 15, n_liquefied),
    np.random.uniform(10, 90, n_liquefied),
    np.random.uniform(0.15, 0.55, n_liquefied),
    np.random.uniform(0, 6, n_liquefied)
])

X = np.vstack([X_stable, X_liquefied])
y = np.array([0] * n_stable + [1] * n_liquefied)  # 0=stable, 1=liquefied

print(f"Dataset: {len(X)} borehole sites (Marmara region)")
print(f"  Stable:    {n_stable} ({n_stable/len(X)*100:.1f}%)")
print(f"  Liquefied: {n_liquefied} ({n_liquefied/len(X)*100:.1f}%)")
print(f"\nFeatures: SPT blow count, depth, fines content, PGA, groundwater depth")

In [ ]:
# A model that ALWAYS predicts "stable"
class AlwaysStableModel:
    def predict(self, X):
        return np.zeros(len(X))

dumb_model = AlwaysStableModel()
y_pred_dumb = dumb_model.predict(X)
accuracy_dumb = accuracy_score(y, y_pred_dumb)
print(f"Accuracy of the 'always stable' model: {accuracy_dumb:.1%}")
print(f"\nBut how many liquefiable sites did it catch?")
print(f"Liquefied sites detected: {np.sum(y_pred_dumb[y == 1]):.0f} out of {n_liquefied}")

In [ ]:
# Let's visualize this disconnect
accuracy = accuracy_dumb * 100
sites_caught = int(np.sum(y_pred_dumb[y == 1] == 1))
total_liq = np.sum(y == 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Accuracy bar
axes[0].bar(['Model Accuracy'], [accuracy], color='steelblue', width=0.4, edgecolor='black', linewidth=1.2)
axes[0].text(0, accuracy + 2, f'{accuracy:.1f}%', ha='center', fontsize=16, fontweight='bold', color='steelblue')
axes[0].set_ylim(0, 110)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Looks Great!', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: Detection bar
axes[1].bar(['Liquefiable Sites\nDetected'], [sites_caught], color='indianred', width=0.4, edgecolor='black', linewidth=1.2)
axes[1].text(0, sites_caught + 2, f'{sites_caught} out of {total_liq}', ha='center', fontsize=16, fontweight='bold', color='indianred')
axes[1].set_ylim(0, total_liq + 10)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('...But Actually Useless', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('The Accuracy Illusion: A "90% Accurate" Model That Catches Nothing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Key Insight: Accuracy is Meaningless When Classes are Imbalanced**
>
> The model above has 90% accuracy — and it's **completely useless**. It missed ALL 100 liquefiable sites.
>
> In geotechnical engineering, soil failures are rare (thankfully). But that makes accuracy a terrible metric for safety-critical applications.

> **[DISCUSS]** Can you think of other engineering scenarios where the "rare event" is the one you care about most? Consider:
> - **Earthquake damage assessment** — most buildings survive, but the damaged ones need urgent attention
> - **Water pipe failure** — most pipes are fine, but a burst main floods entire neighborhoods
> - **Landslide prediction** — most slopes are stable, but a single failure can destroy a highway
> - **Dam safety monitoring** — most readings are normal, but anomalies can precede catastrophic failure

> **[QUICK]** If you were a geotechnical engineer assessing 1,000 borehole sites across the Marmara region and your liquefaction model had 90% accuracy, would you trust it? What additional information would you demand before approving construction on a "stable" site?

---

## 4. The Confusion Matrix

The confusion matrix tells us **how** the model is wrong — not just whether it's wrong. This distinction matters enormously when different types of errors have different consequences.

> **Building the Confusion Matrix Step by Step**
>
> Imagine you have tested 100 borehole sites. You know the truth (from detailed geotechnical investigation after the earthquake) and you have the model's prediction. Let's sort every site into one of four buckets:
>
> 1. Model says "stable," site IS stable — great, safe to build (**True Negative**)
> 2. Model says "liquefiable," site IS stable — unnecessary ground improvement, costs money (**False Positive**)
> 3. Model says "stable," site IS liquefiable — **building on dangerous ground** (**False Negative**)
> 4. Model says "liquefiable," site IS liquefiable — caught it, apply ground improvement (**True Positive**)
>
> The confusion matrix simply counts how many sites fell into each bucket.

In [ ]:
# Let's see this with 10 specific borehole sites
print("Manual Confusion Matrix: Sorting 10 Borehole Sites into Buckets")
print("=" * 65)

# Simulated results for 10 sites
site_ids      = ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']
actual_labels = [0,      0,      1,      0,      0,      1,      0,      0,      0,      0     ]
predicted     = [0,      1,      0,      0,      0,      1,      0,      1,      0,      0     ]
label_map = {0: 'Stable', 1: 'Liquefied'}

tn, fp, fn, tp = 0, 0, 0, 0
for sid, actual, pred in zip(site_ids, actual_labels, predicted):
    if actual == 0 and pred == 0:
        bucket = "True Negative  (correct: stable)"
        tn += 1
    elif actual == 0 and pred == 1:
        bucket = "False Positive (false alarm)"
        fp += 1
    elif actual == 1 and pred == 0:
        bucket = "FALSE NEGATIVE (MISSED LIQUEFACTION!)"
        fn += 1
    else:
        bucket = "True Positive  (caught liquefaction)"
        tp += 1
    print(f"  {sid}: Actual={label_map[actual]:10s}  Predicted={label_map[pred]:10s}  -> {bucket}")

print(f"\nFinal counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"\nNotice: Site S003 was liquefiable but predicted stable — that's the dangerous one!")

> **Definition: Confusion Matrix Quadrants**
>
> | | Predicted Negative | Predicted Positive |
> |---|---|---|
> | **Actually Negative** | True Negative (TN) | False Positive (FP) |
> | **Actually Positive** | False Negative (FN) | True Positive (TP) |
>
> - **TN**: Correctly identified as stable soil
> - **FP**: Stable site flagged as liquefiable ("false alarm" — triggers unnecessary ground improvement)
> - **FN**: Liquefiable site missed ("missed detection" — building on dangerous ground)
> - **TP**: Correctly identified as liquefiable

Now let's use sklearn to compute the confusion matrix on the full dataset. Note the `class_weight='balanced'` parameter — this tells the model to pay extra attention to the minority class (liquefied sites), counteracting the imbalance in our dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Stable', 'Predicted Liquefied'], fontsize=11)
ax.set_yticklabels(['Actually Stable', 'Actually Liquefied'], fontsize=11)
for i in range(2):
    for j in range(2):
        label = f'{cm[i, j]}'
        if i == 0 and j == 0: label += '\n(TN)'
        elif i == 0 and j == 1: label += '\n(FP)'
        elif i == 1 and j == 0: label += '\n(FN)'
        else: label += '\n(TP)'
        ax.text(j, i, label, ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax)
ax.set_title('Confusion Matrix — Soil Liquefaction Classifier', fontsize=13)
plt.tight_layout()
plt.show()

> **Key Insight: In Civil Engineering, Errors are NOT Symmetric**
>
> | Error Type | Meaning | Consequence |
> |---|---|---|
> | **False Negative (FN)** | Liquefiable site predicted as stable | **Building collapses in next earthquake** |
> | **False Positive (FP)** | Stable site predicted as liquefiable | Unnecessary ground improvement (costs money) |
>
> A false negative in liquefaction assessment can be catastrophic — as seen in Adapazarı during the 1999 Izmit earthquake, where buildings on liquefiable soil collapsed. A false positive is merely expensive.

> **Example: The Fire Alarm Analogy**
>
> Think of a fire alarm in your building:
> - **True Positive**: Fire alarm goes off, there IS a fire. Life saved.
> - **False Positive**: Fire alarm goes off, it's just burnt toast. Annoying but harmless.
> - **False Negative**: Fire alarm stays silent during a fire. **Fatal.**
> - **True Negative**: No alarm, no fire. Normal day.
>
> In most safety systems, we tolerate false positives (nuisance alarms) to minimize false negatives (missed dangers). This is exactly the tradeoff in structural health monitoring.

In [ ]:
# Two models, same accuracy, VERY different confusion matrices
# Let's actually build them using different thresholds
y_proba = model.predict_proba(X_test)[:, 1]

# Model A: High threshold (conservative — rarely flags liquefaction)
y_pred_A = (y_proba >= 0.8).astype(int)
cm_A = confusion_matrix(y_test, y_pred_A)
acc_A = accuracy_score(y_test, y_pred_A)

# Model B: Low threshold (aggressive — flags liquefaction more readily)
y_pred_B = (y_proba >= 0.2).astype(int)
cm_B = confusion_matrix(y_test, y_pred_B)
acc_B = accuracy_score(y_test, y_pred_B)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [['TN', 'FP'], ['FN', 'TP']]

for ax, cm_plot, title, acc in zip(axes, [cm_A, cm_B],
    ['Model A: Conservative (high threshold)', 'Model B: Aggressive (low threshold)'],
    [acc_A, acc_B]):
    im = ax.imshow(cm_plot, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred Stable', 'Pred Liquefied'], fontsize=10)
    ax.set_yticklabels(['Actually Stable', 'Actually Liquefied'], fontsize=10)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm_plot[i, j]}\n({labels[i][j]})',
                    ha='center', va='center', fontsize=13, fontweight='bold')
    ax.set_title(f'{title}\nAccuracy: {acc:.1%}', fontsize=11)

plt.suptitle('Which Model Would You Deploy for a Real Construction Project?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the key comparison
liq_caught_A = cm_A[1, 1]
liq_caught_B = cm_B[1, 1]
total_liq = cm_A[1, 0] + cm_A[1, 1]
print(f"Model A — Accuracy: {acc_A:.1%}, Liquefiable sites caught: {liq_caught_A}/{total_liq}")
print(f"Model B — Accuracy: {acc_B:.1%}, Liquefiable sites caught: {liq_caught_B}/{total_liq}")

> **[QUICK]** Sklearn provides a full classification report with precision, recall, and F1 for each class. Let's see what it looks like.

In [ ]:
# Full classification report from sklearn
print("Classification Report — Soil Liquefaction Classifier")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Stable', 'Liquefied']))

> **[PRACTICE]** Look at the classification report above. Identify:
> 1. Which class (Stable or Liquefied) has higher precision?
> 2. Which class has higher recall?
> 3. Why does this make sense given the class imbalance? (Hint: think about what happens when there are very few positive examples — the model has very little to learn from.)

---

## 5. Precision, Recall, and the Tradeoff

Now that we understand the confusion matrix, we can define two critical metrics. But before the math, let's build intuition:

> **Think of a smoke detector:**
> - **Precision** answers: *"When the alarm rings, should I trust it?"* — Of all the times it warned me, how often was there actually a fire?
> - **Recall** answers: *"Am I catching everything dangerous?"* — Of all the real fires, how many did the detector catch?
>
> A very sensitive smoke detector has **high recall** (catches every fire) but **low precision** (also goes off when you cook). A very conservative detector has **high precision** (only alarms on real fires) but **low recall** (might miss a slow-burning fire).

> **Definition: Precision and Recall**
>
> **Precision** = TP / (TP + FP) — *"Of all alarms raised, how many were real?"*
>
> **Recall** = TP / (TP + FN) — *"Of all actual positives, how many did we catch?"*
>
> These two metrics are in tension. Improving one typically hurts the other.

### The Classification Threshold

When a classification model outputs a **probability** (e.g., "there is a 0.73 chance this sample is positive"), we still need to make a **binary decision**: positive or negative. The **threshold** is the cutoff value that converts probabilities into predictions:

$$
\hat{y} = \begin{cases} 1 & \text{if } P(\text{positive}) \geq \text{threshold} \\ 0 & \text{if } P(\text{positive}) < \text{threshold} \end{cases}
$$

By default, most classifiers use a threshold of **0.5**, but this is not always the best choice.

> **Key Insight:** Changing the threshold shifts the balance between **precision** and **recall**:
>
> | Threshold | Effect |
> |-----------|--------|
> | **Low** (e.g., 0.1) | More samples predicted positive → **higher recall** (fewer missed positives) but **lower precision** (more false alarms) |
> | **High** (e.g., 0.9) | Only very confident predictions are positive → **higher precision** but **lower recall** (more missed positives) |
>
> There is no free lunch — improving one metric typically comes at the cost of the other.

> **Example: Civil Engineering** Consider a model that predicts whether a bridge component will fail. A **low threshold** means we flag more components for inspection (costly but safe). A **high threshold** means we only flag components the model is very confident about (cheaper but riskier — we might miss a real failure).

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.95, 0.05)
precisions = []
recalls = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tp = np.sum((y_pred_t == 1) & (y_test == 1))
    fp = np.sum((y_pred_t == 1) & (y_test == 0))
    fn = np.sum((y_pred_t == 0) & (y_test == 1))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    precisions.append(prec)
    recalls.append(rec)

# Show a few key thresholds
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'Interpretation':>40}")
print("-" * 75)
for i in range(0, len(thresholds), 3):
    t = thresholds[i]
    p = precisions[i]
    r = recalls[i]
    interp = "high recall (catch more)" if t < 0.3 else ("balanced" if t < 0.6 else "high precision (fewer false alarms)")
    print(f"{t:>10.2f} {p:>10.2f} {r:>10.2f} {interp:>40}")

In [ ]:
# Visualize the tradeoff at 6 different thresholds
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
threshold_vals = [0.1, 0.2, 0.4, 0.5, 0.7, 0.9]

for ax, t in zip(axes.flat, threshold_vals):
    y_pred_t = (y_proba >= t).astype(int)
    tp = np.sum((y_pred_t == 1) & (y_test == 1))
    fp = np.sum((y_pred_t == 1) & (y_test == 0))
    fn = np.sum((y_pred_t == 0) & (y_test == 1))
    tn = np.sum((y_pred_t == 0) & (y_test == 0))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Plot bars for TP, FP, FN, TN
    categories = ['TN', 'FP', 'FN', 'TP']
    values = [tn, fp, fn, tp]
    colors = ['steelblue', 'sandybrown', 'indianred', 'mediumseagreen']
    ax.bar(categories, values, color=colors, edgecolor='black', linewidth=0.8)
    for i_bar, v in enumerate(values):
        if v > 0:
            ax.text(i_bar, v + 0.3, str(v), ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'Threshold = {t:.1f}\nPrec={prec:.2f}, Recall={rec:.2f}', fontsize=11)
    ax.set_ylim(0, max(tn + 5, 20))
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('How Changing the Threshold Affects the Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **[TOGETHER]** Look at the 6 plots above. As the threshold moves from low (0.1) to high (0.9):
> 1. What happens to the number of false alarms (FP)?
> 2. What happens to the number of missed detections (FN)?
> 3. At which threshold would you feel comfortable deploying this model for a real construction project?
> 4. Is there a threshold that makes BOTH FP and FN zero? Why or why not?

### The Precision–Recall Tradeoff in Detail

Both metrics share **TP** in the numerator, but their denominators pull in opposite directions:

- To **increase recall**, we lower the threshold so we catch more true positives — but this also lets in more false positives, which **decreases precision**.
- To **increase precision**, we raise the threshold so we only predict positive when we are very confident — but this causes us to miss some true positives, which **decreases recall**.

The only way to improve both simultaneously is to build a **better model** (one that assigns higher probabilities to true positives and lower probabilities to true negatives).

#### The F1 Score: A Single Balanced Metric

When neither precision nor recall alone captures what we care about, the **F1 score** provides a single number. But why not just average them?

Consider a model with **Precision = 1.0** and **Recall = 0.01** (almost never predicts positive, but when it does, it's always right):

$$
\text{Arithmetic mean} = \frac{1.0 + 0.01}{2} = 0.505 \quad \text{(looks decent!)}
$$

$$
F_1 = 2 \cdot \frac{1.0 \times 0.01}{1.0 + 0.01} = 0.02 \quad \text{(correctly shows this model is nearly useless)}
$$

The **harmonic mean** penalizes extreme imbalances — the F1 score is high only when **both** metrics are reasonably high.

| F1 Score | Interpretation |
|----------|---------------|
| **1.0** | Perfect — no false positives and no false negatives |
| **> 0.8** | Strong classifier for most practical applications |
| **0.5 – 0.8** | Moderate — may need tuning depending on the application |
| **< 0.5** | Weak — the model is making many errors of one type or both |

#### Weighted Variants: $F_\beta$

Sometimes false negatives and false positives are **not equally costly**. The generalized $F_\beta$ score lets you control the balance:

$$
F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \times \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}
$$

| Variant | $\beta$ | Emphasis | Use case |
|---------|---------|----------|----------|
| **$F_{0.5}$** | 0.5 | Weighs **precision** more | When false alarms are costly (e.g., dispatching emergency crews) |
| **$F_1$** | 1.0 | Equal weight | Default balanced metric |
| **$F_2$** | 2.0 | Weighs **recall** more | When missing positives is dangerous (e.g., structural failure screening) |

> **Key Insight:** The F1 score is the best **default** when you have no strong reason to prefer precision over recall. In real engineering applications, always think about what type of error is more costly before choosing your evaluation metric.

In [ ]:
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_curve, precision_curve, 'b-', linewidth=2)
ax.set_xlabel('Recall (How many liquefiable sites we catch)', fontsize=12)
ax.set_ylabel('Precision (How many alarms are real)', fontsize=12)
ax.set_title('Precision-Recall Tradeoff', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

> **Key Insight: You Can't Maximize Both**
> - **Optimize Recall** when missing a positive is catastrophic → structural safety monitoring
> - **Optimize Precision** when false alarms are costly → routine maintenance scheduling
> - **F1 Score** = harmonic mean of precision and recall — use when you want one number that balances both

> **Example: Civil Engineering Applications**
> - **Liquefaction screening** → High recall (don't miss a dangerous site, even if you trigger some unnecessary ground improvement)
> - **Routine pipe inspection** → High precision (don't waste crew time inspecting healthy pipes)
> - **Emergency earthquake response** → High recall (better to over-evacuate than miss a damaged building)

> **Example: Flood Early Warning System**
>
> You build a model that predicts whether a river will flood in the next 24 hours.
> - **High recall strategy** (threshold = 0.2): You evacuate neighborhoods whenever there is even a 20% chance of flooding. Many false evacuations, but you never miss a real flood. Cost: public trust erodes over time from false alarms.
> - **High precision strategy** (threshold = 0.8): You only evacuate when the model is 80%+ confident. Fewer unnecessary evacuations, but you might miss a real flood. Cost: lives at risk.
> - **The tradeoff is real and there is no "correct" answer — it depends on the consequences.**

> **[PRACTICE]** Let's compute precision, recall, and F1 manually for our liquefaction model to reinforce the formulas.

In [ ]:
# Compute F1 score manually
# F1 = 2 * (precision * recall) / (precision + recall)

tp = np.sum((y_pred == 1) & (y_test == 1))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_val = 2 * (precision_val * recall_val) / (precision_val + recall_val) if (precision_val + recall_val) > 0 else 0

print(f"Precision: {precision_val:.3f}")
print(f"Recall:    {recall_val:.3f}")
print(f"F1 Score:  {f1_val:.3f}")
print()
if recall_val > precision_val:
    print("This model leans toward recall — it catches more liquefiable sites")
    print("but raises more false alarms. For geotechnical safety, this is usually preferred.")
else:
    print("This model leans toward precision — its alarms are more trustworthy")
    print("but it may miss some liquefiable sites. For safety applications, be cautious.")

> **[TOGETHER]** Would you accept this F1 score for a liquefaction screening system? What F1 score would you consider the minimum for deploying such a model in practice?

---

## 6. ROC Curves and AUC

So far, we've evaluated our model at a **specific threshold**. But what if we want to compare two models across **all possible thresholds** simultaneously? That's what the ROC curve does.

The **Receiver Operating Characteristic (ROC)** curve gives us a way to evaluate a classifier across every threshold at once. The **Area Under the Curve (AUC)** summarizes this into a single number — think of it as a **report card GPA** for your model: it doesn't tell you how the model performed on any single exam (threshold), but it summarizes the overall performance.

> **Building the ROC Curve One Point at a Time**
>
> Remember that **Recall** (also called True Positive Rate) answers: "Of all liquefiable sites, how many did we catch?"
>
> Now we need one more metric: **False Positive Rate (FPR)** = FP / (FP + TN) — *"Of all stable sites, how many did we falsely flag?"*
>
> Here's how the ROC curve is built:
>
> 1. At threshold = 0.5, our liquefaction model produces a specific confusion matrix. From that matrix, compute TPR and FPR. That gives us **one point** on the ROC plot: (FPR, TPR).
> 2. Change the threshold to 0.3. New confusion matrix → new TPR and FPR → **another point**.
> 3. Repeat for every possible threshold from 0 to 1. Connect the dots.
> 4. **That's the ROC curve.**

In [ ]:
# Let's build the ROC curve point by point
print("Building the ROC curve step by step:")
print("=" * 70)

demo_thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
roc_points = []

for t in demo_thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tp = np.sum((y_pred_t == 1) & (y_test == 1))
    fp = np.sum((y_pred_t == 1) & (y_test == 0))
    fn = np.sum((y_pred_t == 0) & (y_test == 1))
    tn = np.sum((y_pred_t == 0) & (y_test == 0))
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    roc_points.append((fpr, tpr))
    print(f"\nThreshold = {t:.1f}:")
    print(f"  Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"  FPR = {fp}/({fp}+{tn}) = {fpr:.3f}  |  TPR = {tp}/({tp}+{fn}) = {tpr:.3f}")
    print(f"  → Plot point: ({fpr:.3f}, {tpr:.3f})")

# Now plot these individual points AND the full smooth curve
fpr_full, tpr_full, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_full, tpr_full, color='steelblue', linewidth=1.5, alpha=0.4, label='Full ROC curve')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random guessing')

for i, (fpr_pt, tpr_pt) in enumerate(roc_points):
    t = demo_thresholds[i]
    ax.scatter(fpr_pt, tpr_pt, s=120, zorder=5, edgecolors='black', linewidth=1.5)
    ax.annotate(f't={t}', (fpr_pt, tpr_pt), textcoords="offset points",
                xytext=(10, -10), fontsize=10, fontweight='bold')

ax.set_xlabel('False Positive Rate (stable sites falsely flagged)', fontsize=12)
ax.set_ylabel('True Positive Rate = Recall (liquefiable sites caught)', fontsize=12)
ax.set_title('ROC Curve Built Point by Point', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEach dot is one threshold. Connect them all → you get the ROC curve!")

### Comparing Multiple Models with ROC

One of the best uses of ROC curves is comparing different classifiers on the same problem. Let's try three models:

- **Logistic Regression** — fits a linear decision boundary (the model we've been using)
- **Decision Tree** — splits data using a sequence of if/else rules on individual features
- **K-Nearest Neighbors (KNN)** — classifies a sample based on the majority vote of its closest neighbors in feature space

We won't dive deep into how these models work — that's for future weeks. For now, the point is that different models can produce very different ROC curves on the same data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Train multiple models on the liquefaction data
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guessing (AUC=0.50)')

for name, m in models.items():
    m.fit(X_train, y_train)
    if hasattr(m, 'predict_proba'):
        y_score = m.predict_proba(X_test)[:, 1]
    else:
        y_score = m.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={roc_auc:.2f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Comparing Liquefaction Models', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Key Insight: AUC Tells You How Well Your Model Separates Classes**
> - AUC = 1.0: Perfect separation (every liquefiable site ranked higher than every stable site)
> - AUC = 0.5: Random guessing (the diagonal — no better than flipping a coin)
> - Higher AUC = better model, regardless of threshold choice
>
> Think of it like a GPA: it summarizes overall performance without telling you the score on any single test.

In [ ]:
# Compare AUC values numerically
print("Liquefaction Model Comparison Summary")
print("=" * 55)
for name, m in models.items():
    y_pred_m = m.predict(X_test)
    if hasattr(m, 'predict_proba'):
        y_score_m = m.predict_proba(X_test)[:, 1]
    else:
        y_score_m = m.decision_function(X_test)
    fpr_m, tpr_m, _ = roc_curve(y_test, y_score_m)
    roc_auc_m = auc(fpr_m, tpr_m)
    acc_m = accuracy_score(y_test, y_pred_m)
    print(f"{name:30s}  Accuracy={acc_m:.3f}  AUC={roc_auc_m:.3f}")

> **[DISCUSS]** A colleague tells you: "My liquefaction model has AUC = 0.95, so it's great." What questions would you ask before trusting this claim for a real construction project?
>
> Consider:
> - How imbalanced are the classes? (Most sites don't liquefy)
> - What specific threshold will they deploy at?
> - What kind of errors matter most — missing a liquefiable site, or triggering unnecessary ground improvement?
> - Was the AUC computed on truly unseen test data, or on sites from the same earthquake event?

> **Key Insight: When to Use Which Curve**
> - **ROC curve**: Best when classes are roughly balanced. The FPR axis can be misleading when negatives vastly outnumber positives (a small FPR can still mean many false alarms).
> - **Precision-Recall curve**: Better for imbalanced datasets (like our liquefaction data with 90% stable sites). It focuses on the minority class.
> - For most geotechnical safety problems, the **precision-recall curve is more informative** than ROC.

---

## 7. Overfitting — When Your Model Memorizes

> **Analogy: Studying for an Exam**
>
> Imagine two students preparing for a civil engineering exam:
> - **Student A** memorizes every answer from past exams word-for-word. On a practice test using those same past questions, they score 100%. But on the real exam with new questions, they fail.
> - **Student B** studies the underlying concepts (equilibrium, compatibility, material behavior). They score 85% on practice tests but also 82% on the real exam.
>
> Student A is **overfitting**. Student B is **generalizing**. We want our models to be Student B.

A model that performs perfectly on training data but poorly on new data has **memorized** the noise in the training set rather than learning the underlying pattern. This is one of the most fundamental challenges in machine learning.

In [ ]:
# Simple memorization demo: 5 speed-density measurements
np.random.seed(42)

# Training data: 5 traffic sensor readings (vehicles/km → speed km/h)
density_train = np.array([10, 30, 60, 90, 120])
speed_train = np.array([105, 88, 55, 25, 8]) + np.random.normal(0, 2, 5)

# Fit a degree-4 polynomial (passes through all 5 points exactly)
coeffs = np.polyfit(density_train, speed_train, 4)

# Test on 3 new sensor readings
density_test = np.array([20, 50, 100])
# True speeds from the Underwood model
speed_test = 110 * np.exp(-density_test / 70)
speed_pred = np.polyval(coeffs, density_test)

train_error = np.mean((speed_train - np.polyval(coeffs, density_train))**2)
test_error = np.mean((speed_test - speed_pred)**2)

print("Memorization Demo: Traffic Speed-Density")
print("=" * 45)
print(f"Training error (5 sensors):  {train_error:.6f}  <- practically zero!")
print(f"Test error (3 new sensors):  {test_error:.4f}  <- much worse!")
print(f"\nThe model memorized the 5 training readings perfectly,")
print(f"but fails on new traffic data it hasn't seen before.")

In [ ]:
np.random.seed(42)
n_pts = 30

# Traffic speed-density relationship (Underwood model: speed = v_f * exp(-density/k_opt))
# v_f = 110 km/h (free-flow speed), k_opt = 70 veh/km (optimum density)
x = np.sort(np.random.uniform(5, 130, n_pts))  # density (vehicles/km)
y_true = 110 * np.exp(-x / 70)                  # true speed (km/h)
y_noisy = y_true + np.random.normal(0, 6, n_pts) # noisy measurements

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
x_plot = np.linspace(5, 130, 300)

for ax, degree in zip(axes, [1, 3, 10, 20]):
    coeffs = np.polyfit(x, y_noisy, degree)
    y_fit = np.polyval(coeffs, x_plot)

    ax.scatter(x, y_noisy, s=40, color='steelblue', edgecolors='k', zorder=3)
    ax.plot(x_plot, y_fit, 'r-', linewidth=2)
    ax.plot(x_plot, 110 * np.exp(-x_plot / 70), 'g--', linewidth=1, alpha=0.5, label='True relationship')
    ax.set_title(f'Degree {degree}', fontsize=12)
    ax.set_ylim(-20, 130)
    ax.set_xlabel('Density (veh/km)', fontsize=9)
    ax.set_ylabel('Speed (km/h)', fontsize=9)
    ax.grid(True, alpha=0.3)
    if degree == 1: ax.legend(fontsize=9)

plt.suptitle('Polynomial Fits: From Underfitting to Overfitting (Traffic Speed-Density)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Definition: Underfitting vs. Overfitting**
>
> - **Underfitting** (degree 1): A straight line cannot capture the exponential decay of speed with density. High bias, low variance.
> - **Just right** (degree 3): Model captures the smooth decay without fitting sensor noise.
> - **Overfitting** (degree 20): Model chases every noisy speed reading. Low bias, high variance.
>
> The goal is to find the **sweet spot** — complex enough to capture real patterns, simple enough to generalize.

### Why Train-Test Split?

The polynomial plots above used all 30 data points for both fitting *and* evaluation. That's like grading a student on the exact questions they practiced — of course they'll do well.

To get an honest evaluation, we need **unseen data**: data the model has never seen during training.

> **Definition: Train-Test Split**
>
> Divide your dataset into two parts:
> - **Training set** (~70-80%): used to fit the model
> - **Test set** (~20-30%): held back, used *only* for evaluation
>
> **Never** use test data during training or hyperparameter tuning. The test set estimates how well your model will perform on truly new data.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))

# Training data rectangle
ax.barh(0, 80, left=0, height=0.5, color='steelblue', alpha=0.4, edgecolor='k')
ax.text(40, 0, 'Training Data (80%)', ha='center', va='center', fontsize=13, fontweight='bold')

# Test data rectangle
ax.barh(0, 20, left=80, height=0.5, color='indianred', alpha=0.4, edgecolor='k')
ax.text(90, 0, 'Test (20%)', ha='center', va='center', fontsize=13, fontweight='bold')

ax.set_xlim(-2, 102)
ax.set_ylim(-0.6, 0.6)
ax.set_xlabel('All Available Data', fontsize=12)
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Split data
x_train_ov, x_test_ov, y_train_ov, y_test_ov = train_test_split(
    x, y_noisy, test_size=0.3, random_state=42
)

degrees = range(1, 16)
train_errors = []
test_errors = []

for d in degrees:
    pipe = make_pipeline(PolynomialFeatures(d), LinearRegression())
    pipe.fit(x_train_ov.reshape(-1, 1), y_train_ov)

    train_pred = pipe.predict(x_train_ov.reshape(-1, 1))
    test_pred = pipe.predict(x_test_ov.reshape(-1, 1))

    train_errors.append(np.sqrt(mean_squared_error(y_train_ov, train_pred)))
    test_errors.append(np.sqrt(mean_squared_error(y_test_ov, test_pred)))

# Cap very large test errors for visualization (values above 5 are truncated)
test_errors_capped = [min(e, 5) for e in test_errors]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(degrees), train_errors, 'o-', color='steelblue', linewidth=2, label='Training Error (RMSE)')
ax.plot(list(degrees), test_errors_capped, 's-', color='indianred', linewidth=2, label='Test Error (RMSE)')
ax.axvline(x=3, color='green', linestyle='--', alpha=0.5, label='Sweet spot')
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Train vs Test Error: The Overfitting Curve', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
if max(test_errors) > 5:
    ax.annotate('(test errors above 5 truncated)', xy=(0.98, 0.98),
                xycoords='axes fraction', ha='right', va='top', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

> **[TOGETHER]** Look at the U-curve above:
> 1. At degree 1, both training error and test error are high. What does that mean?
> 2. Around degree 3, test error is lowest. What does that mean?
> 3. Beyond degree 8, training error approaches zero but test error explodes. What does that mean?
> 4. The green dashed line marks the sweet spot. In practice, how do you find this spot?

> **Key Insight: A Model That's Perfect on Training Data is Suspicious, Not Impressive**
> - Training error always decreases with complexity
> - Test error decreases, then **increases** (the U-curve)
> - The gap between train and test error = overfitting

> **Example: Civil Engineering**
>
> You collect speed-density data from 50 highway sensors during one week on the O-4 motorway near Istanbul. Your degree-12 polynomial fits the training data beautifully (R² = 0.99).
>
> Then you deploy it for the following week's traffic management system. At moderate densities, your model predicts **negative speeds** — a physical impossibility. The polynomial oscillations that fit the noise in training data produce absurd predictions on new data.
>
> The model memorized the specific traffic patterns of that one week, not the general physics of traffic flow. **This is overfitting in the real world.**

> **[TOGETHER]** Let's examine the exact train and test error numbers at each polynomial degree. Watch how the status changes from underfitting to good fit to overfitting.

In [ ]:
# Let's look at the numbers
print(f"{'Degree':>8} {'Train RMSE':>12} {'Test RMSE':>12} {'Gap':>10} {'Status':>15}")
print("-" * 60)
for d, tr_e, te_e in zip(degrees, train_errors, test_errors):
    gap = te_e - tr_e
    if tr_e > 3.0:
        status = "underfitting"
    elif gap < 0.5:
        status = "good fit"
    elif gap < 2.0:
        status = "overfitting"
    else:
        status = "SEVERE overfit"
    te_display = min(te_e, 99.99)
    print(f"{d:>8d} {tr_e:>12.4f} {te_display:>12.4f} {gap:>10.4f} {status:>15}")

### Cross-Validation: A More Robust Estimate

The U-curve above depends on *which* 30% of data ended up in the test set. A different random split might give a different "best degree." How do we get a more reliable answer?

> **Definition: K-Fold Cross-Validation**
>
> 1. Split data into $k$ equal parts (folds)
> 2. Train on $k-1$ folds, test on the remaining fold
> 3. Repeat $k$ times — each fold serves as the test set once
> 4. Average the $k$ scores for a robust performance estimate
>
> **Why it works:** Every data point gets used for both training and testing, and the average smooths out the luck of any single split.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

n_folds = 5
colors_train = 'steelblue'
colors_test = 'indianred'

for i in range(n_folds):
    y_pos = n_folds - 1 - i
    for j in range(n_folds):
        x_start = j * (1.0 / n_folds)
        width = 1.0 / n_folds
        if j == i:
            color = colors_test
            alpha = 0.5
            label = 'Test'
        else:
            color = colors_train
            alpha = 0.4
            label = 'Train'
        rect = plt.Rectangle((x_start, y_pos - 0.35), width, 0.7,
                             facecolor=color, alpha=alpha, edgecolor='k', linewidth=1)
        ax.add_patch(rect)
        if j == i:
            ax.text(x_start + width / 2, y_pos, 'Test', ha='center', va='center',
                    fontsize=10, fontweight='bold')
    ax.text(-0.08, y_pos, f'Fold {i+1}:', ha='right', va='center', fontsize=11)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors_train, alpha=0.4, edgecolor='k', label='Train'),
                   Patch(facecolor=colors_test, alpha=0.5, edgecolor='k', label='Test')]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

ax.set_xlim(-0.15, 1.05)
ax.set_ylim(-0.6, n_folds - 0.2)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('5-Fold Cross-Validation', fontsize=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation on our polynomial regression problem
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Use the polynomial data from the overfitting demo above
print("Cross-Validation Scores by Polynomial Degree")
print("=" * 55)
print(f"{'Degree':>8} {'Mean R²':>10} {'Std Dev':>10} {'Verdict':>15}")
print("-" * 55)

for degree in [1, 2, 3, 5, 10]:
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    scores = cross_val_score(pipe, x.reshape(-1, 1), y_noisy, cv=5, scoring='r2')

    if scores.mean() > 0.7 and scores.std() < 0.15:
        verdict = "Good"
    elif scores.mean() < 0.3:
        verdict = "Underfitting"
    else:
        verdict = "Overfitting"

    print(f"{degree:>8d} {scores.mean():>10.3f} {scores.std():>10.3f} {verdict:>15}")

print()
print("Note: High std dev across folds = model is sensitive to which data it sees = overfitting")

> **Key Insight: Cross-Validation is Your Defense Against Overfitting**
>
> Cross-validation is precisely the tool you use to detect overfitting in practice: if your model performs well on all K folds (not just the training set), you have evidence that it generalizes. If performance varies wildly across folds, your model may be overfitting to specific subsets of the data.

---

## 8. Data Leakage — The Silent Killer

Data leakage occurs when information from outside the training dataset is used to create the model. It leads to **overly optimistic performance estimates** that collapse in production.

This is arguably the most common and most dangerous mistake in applied machine learning.

> **Analogy: The Exam Answer Key**
>
> Imagine you're studying for a final exam and you accidentally find the answer key beforehand. You ace the exam with a perfect score. But you didn't actually learn anything — your "performance" is an illusion.
>
> On the next exam (with different questions), you fail miserably because the leaked answers are no longer available.
>
> Data leakage is the same: your model "saw the answers" during training, so it looks brilliant. But in production (the next exam), it fails.

In [ ]:
np.random.seed(42)
n = 200

# Earthquake building damage data — inspired by the 2023 Kahramanmaraş earthquake
n_floors = np.random.randint(1, 10, n).astype(float)
building_age = np.random.uniform(0, 80, n)
soil_score = np.random.uniform(1, 5, n)  # 1=rock, 5=soft clay

# True damage score (0-100) based on pre-earthquake features
damage_score = (5 * n_floors + 0.4 * building_age + 8 * soil_score
                + np.random.normal(0, 10, n))
damage_score = np.clip(damage_score, 0, 100)

# Post-earthquake observation — only measurable AFTER the event!
observed_crack_width = 0.5 * damage_score + np.random.normal(0, 3, n)
observed_crack_width = np.clip(observed_crack_width, 0, None)

df_leak = pd.DataFrame({
    'n_floors': n_floors,
    'building_age': building_age,
    'soil_score': soil_score,
    'crack_width_mm': observed_crack_width,  # LEAKED — post-earthquake measurement!
    'damage_score': damage_score  # TARGET
})

print("Features available:")
print(df_leak.columns.tolist())
print("\n'crack_width_mm' was measured AFTER the earthquake — this is data leakage!")
print("You can't know crack widths before the earthquake happens.")

In [ ]:
# WITH leakage (including post-earthquake crack width)
X_leaked = df_leak[['n_floors', 'building_age', 'soil_score', 'crack_width_mm']].values
y_target = df_leak['damage_score'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_leaked, y_target, test_size=0.2, random_state=42)
model_leaked = LinearRegression().fit(X_tr, y_tr)

# WITHOUT leakage (only pre-earthquake features)
X_clean = df_leak[['n_floors', 'building_age', 'soil_score']].values
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_clean, y_target, test_size=0.2, random_state=42)
model_clean = LinearRegression().fit(X_tr2, y_tr2)

print(f"WITH leaked feature (crack width):  R² = {model_leaked.score(X_te, y_te):.3f}  <- Too good to be true!")
print(f"WITHOUT leaked feature:             R² = {model_clean.score(X_te2, y_te2):.3f}  <- The real performance")
print(f"\nThe leaked model looks amazing but would fail in practice,")
print(f"because you can't measure crack widths before an earthquake.")

In [ ]:
# Visualize: Predicted vs Actual for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Leaked model
y_pred_leaked = model_leaked.predict(X_te)
axes[0].scatter(y_te, y_pred_leaked, alpha=0.6, color='indianred', edgecolors='k', s=50)
axes[0].plot([0, 100], [0, 100], 'k--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Damage Score', fontsize=11)
axes[0].set_ylabel('Predicted Damage Score', fontsize=11)
axes[0].set_title(f'WITH Leaked Feature (R² = {model_leaked.score(X_te, y_te):.3f})\nSuspiciously perfect!', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Clean model
y_pred_clean = model_clean.predict(X_te2)
axes[1].scatter(y_te2, y_pred_clean, alpha=0.6, color='steelblue', edgecolors='k', s=50)
axes[1].plot([0, 100], [0, 100], 'k--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual Damage Score', fontsize=11)
axes[1].set_ylabel('Predicted Damage Score', fontsize=11)
axes[1].set_title(f'WITHOUT Leaked Feature (R² = {model_clean.score(X_te2, y_te2):.3f})\nRealistic scatter', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Data Leakage: The Difference is Visible', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Key Insight: Data Leakage Gives You False Confidence**
>
> Always ask: *"Would I have this feature at the time I need to make my prediction?"*
>
> **3 Common Leakage Patterns:**
> 1. **Target leakage**: A feature derived from or strongly correlated with the target. *Example: Using "observed crack width" (measured after the earthquake) to predict earthquake damage.*
> 2. **Train-test contamination**: Scaling/normalizing before splitting. *Example: Computing mean building height across ALL buildings (including test set) to normalize.*
> 3. **Temporal leakage**: Using future data to predict the past. *Example: Using 2024 repair records to predict which buildings were damaged in 2023.*

> **[PRACTICE]** Let's see what happens when we accidentally let test data statistics leak into training through feature scaling.

In [ ]:
# Train-test contamination example
from sklearn.preprocessing import StandardScaler

# WRONG way: scale before split (test data statistics leak into training)
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X_clean)  # fit on ALL data
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(X_scaled_wrong, y_target, test_size=0.2, random_state=42)
model_wrong = LinearRegression().fit(Xw_tr, yw_tr)

# RIGHT way: scale after split (only use training data statistics)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_clean, y_target, test_size=0.2, random_state=42)
scaler_right = StandardScaler()
Xr_tr_scaled = scaler_right.fit_transform(Xr_tr)     # fit ONLY on training
Xr_te_scaled = scaler_right.transform(Xr_te)          # transform test with train stats
model_right = LinearRegression().fit(Xr_tr_scaled, yr_tr)

print("Train-Test Contamination Demo")
print("=" * 50)
print(f"WRONG (scale before split): R² = {model_wrong.score(Xw_te, yw_te):.4f}")
print(f"RIGHT (scale after split):  R² = {model_right.score(Xr_te_scaled, yr_te):.4f}")
print(f"\nIn this case the difference is small because we have few features")
print(f"and plenty of data. With more features and less data, contamination")
print(f"can drastically inflate scores.")

> **Example: Civil Engineering — The Timeline Test**
>
> Imagine you are predicting building damage in a future earthquake. Ask yourself: *"When does each feature become available?"*
>
> | Feature | Available before the earthquake? | Why? |
> |---|---|---|
> | Number of floors | Yes | Known from building records |
> | Building age | Yes | Known from permit records |
> | Soil type at site | Yes | Known from geological surveys |
> | Observed crack width | **No** | Only measurable after the earthquake |
> | Differential settlement | **No** | Only measurable after the earthquake |
> | Repair cost estimate | **No — this is essentially the target!** | Circular reasoning |
>
> This is directly relevant to Turkey's post-earthquake building screening program (Law 6306). The goal is to identify at-risk buildings **before** the next earthquake — so only pre-earthquake features are valid.

> **[PRACTICE]** You are building a model to predict landslide risk along a highway corridor in the Black Sea region. Your dataset includes:
> - `slope_angle`, `elevation`, `rainfall_monthly`, `vegetation_index`, `soil_type`, `distance_to_road`
> - `road_closure_status` (whether the road was closed due to a landslide)
> - `landslide_occurred` (target: yes/no)
>
> Which feature(s) should you NOT use? Why?
>
> *(Hint: Apply the timeline test — would you have this feature BEFORE the landslide happens?)*

---

## 9. Putting It Together

### Model Evaluation Checklist

Before you report results, ask yourself:

| # | Question | If "No", Then... |
|---|---|---|
| 1 | Is my dataset balanced? | Accuracy is misleading — use precision, recall, F1 |
| 2 | Do I know the cost of each error type? | Define: What's worse, FP or FN? |
| 3 | Am I evaluating on truly unseen data? | Use train-test split or cross-validation |
| 4 | Does my model generalize? | Check train vs test error gap |
| 5 | Could there be data leakage? | Audit every feature: would I have it at prediction time? |

### Mini Capstone: Water Pipe Failure Prediction

Istanbul's water utility **ISKI** manages one of the largest urban pipe networks in Europe — over 20,000 km of water mains serving 16 million people. Aging infrastructure means pipe failures are inevitable, but inspecting every pipe is impossible.

You've been hired to build a model that predicts which pipe segments are most likely to fail in the next year. Your dataset: **5,000 pipe segments** — 4,750 operational and 250 with recorded failures.

**Let's walk through the full evaluation pipeline with this scenario.**

In [ ]:
# Generate the water pipe failure dataset
np.random.seed(42)

n_pipes = 5000
n_operational = 4750
n_failed = 250

# Features for operational pipes
X_oper = np.column_stack([
    np.random.uniform(5, 80, n_operational),       # pipe_age (years)
    np.random.uniform(100, 600, n_operational),    # diameter_mm
    np.random.uniform(5, 500, n_operational),      # length_m
    np.random.choice([0, 1, 2, 3, 4], n_operational, p=[0.15, 0.25, 0.30, 0.20, 0.10]),  # material (0=cast iron, 1=ductile iron, 2=PVC, 3=steel, 4=concrete)
    np.random.uniform(2, 8, n_operational),        # pressure_bar
    np.random.choice([1, 2, 3], n_operational, p=[0.4, 0.4, 0.2]),  # soil_corrosivity (1=low, 2=medium, 3=high)
    np.random.uniform(0.8, 3.0, n_operational),    # burial_depth_m
    np.random.poisson(1, n_operational)            # prev_breaks
])

# Features for failed pipes (older, smaller, cast iron, more corrosive soil, more breaks)
X_fail = np.column_stack([
    np.random.uniform(40, 120, n_failed),          # older pipes
    np.random.uniform(75, 300, n_failed),          # smaller diameter
    np.random.uniform(5, 500, n_failed),           # length_m
    np.random.choice([0, 1, 2, 3, 4], n_failed, p=[0.45, 0.25, 0.10, 0.15, 0.05]),  # mostly cast iron
    np.random.uniform(3, 8, n_failed),             # pressure_bar
    np.random.choice([1, 2, 3], n_failed, p=[0.15, 0.35, 0.50]),  # more corrosive soil
    np.random.uniform(0.8, 2.5, n_failed),         # burial_depth_m
    np.random.poisson(4, n_failed)                 # more previous breaks
])

X_pipe = np.vstack([X_oper, X_fail])
y_pipe = np.array([0] * n_operational + [1] * n_failed)

# Shuffle
shuffle_idx = np.random.permutation(n_pipes)
X_pipe, y_pipe = X_pipe[shuffle_idx], y_pipe[shuffle_idx]

feature_names = ['pipe_age', 'diameter_mm', 'length_m', 'material', 'pressure_bar',
                 'soil_corrosivity', 'burial_depth_m', 'prev_breaks']

print(f"Dataset: {n_pipes} pipe segments ({n_operational} operational, {n_failed} failed)")
print(f"Class balance: {n_failed/n_pipes*100:.1f}% failed")
print(f"Features: {', '.join(feature_names)}")

In [ ]:
# Train model and evaluate
X_pipe_train, X_pipe_test, y_pipe_train, y_pipe_test = train_test_split(
    X_pipe, y_pipe, test_size=0.3, random_state=42, stratify=y_pipe
)

pipe_model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
pipe_model.fit(X_pipe_train, y_pipe_train)
y_pipe_pred = pipe_model.predict(X_pipe_test)
y_pipe_proba = pipe_model.predict_proba(X_pipe_test)[:, 1]

# Results
acc_pipe = accuracy_score(y_pipe_test, y_pipe_pred)
cm_pipe = confusion_matrix(y_pipe_test, y_pipe_pred)

print("Water Pipe Failure Prediction — Model Results")
print("=" * 60)
print(f"\nAccuracy: {acc_pipe:.1%}")
print(f"\nConfusion Matrix:")
print(f"  TN={cm_pipe[0,0]:4d}  FP={cm_pipe[0,1]:4d}")
print(f"  FN={cm_pipe[1,0]:4d}  TP={cm_pipe[1,1]:4d}")
print(f"\nClassification Report:")
print(classification_report(y_pipe_test, y_pipe_pred, target_names=['Operational', 'Failed']))

In [ ]:
# ROC curve for the pipe failure model
fpr_pipe, tpr_pipe, _ = roc_curve(y_pipe_test, y_pipe_proba)
auc_pipe = auc(fpr_pipe, tpr_pipe)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_pipe, tpr_pipe, color='steelblue', linewidth=2, label=f'Model (AUC={auc_pipe:.2f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guessing')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Water Pipe Failure Model', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **[TOGETHER]** Looking at the pipe failure model results above:
> 1. Is the accuracy misleadingly high? (Check the class balance — 95% of pipes are operational.)
> 2. What does the confusion matrix tell you about false negatives? How many pipe failures did we miss?
> 3. For this application (water infrastructure), should we optimize for precision or recall? Consider: a missed pipe failure can flood a neighborhood, but sending a crew to a healthy pipe wastes resources.
> 4. If ISKI can only inspect 500 pipe segments this year, how would you use the model's probability scores to prioritize?

> **[TOGETHER]** If ISKI can inspect 500 pipe segments this year, which threshold gives the best balance between catching failures and not wasting crew time?

In [ ]:
# Threshold analysis: What happens at different decision thresholds?
print("Threshold Analysis — Water Pipe Failure Model")
print("=" * 75)
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'FP (false alarms)':>18} {'FN (missed)':>12}")
print("-" * 75)

for t in [0.1, 0.2, 0.3, 0.5, 0.7]:
    y_t = (y_pipe_proba >= t).astype(int)
    tp = np.sum((y_t == 1) & (y_pipe_test == 1))
    fp = np.sum((y_t == 1) & (y_pipe_test == 0))
    fn = np.sum((y_t == 0) & (y_pipe_test == 1))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"{t:>10.1f} {prec:>10.2f} {rec:>10.2f} {fp:>18d} {fn:>12d}")

print(f"\nTotal failed pipes in test set: {np.sum(y_pipe_test)}")
print(f"Total operational pipes in test set: {np.sum(y_pipe_test == 0)}")

In [ ]:
# Self-check quiz
print("Quick Self-Check")
print("=" * 50)
print()
print("Q1: Your pipe failure model has 95% accuracy.")
print("    Should you celebrate? Why or why not?")
print("    # Your answer: ")
print("    # Hint: What percentage of pipes are operational?")
print()
print("Q2: Your model has perfect R² on training data.")
print("    What should you suspect?")
print("    # Your answer: ")
print("    # Hint: Think about Student A vs Student B from Section 5")
print()
print("Q3: You're predicting water pipe failure. Which is worse:")
print("    predicting 'operational' when it's about to burst (FN), or")
print("    sending a crew to an intact pipe (FP)?")
print("    # Your answer: ")
print("    # Hint: What are the consequences of each error?")
print()
print("Q4: Your model uses 'repair_cost' as a feature. This cost")
print("    was recorded AFTER the pipe failed. Is this a problem?")
print("    # Your answer: ")
print("    # Hint: Apply the timeline test from Section 6")

### Connection to the CE49X Course

This notebook gives you the evaluation framework for:
- **Week 07** (Naive Bayes): Before reporting "my Naive Bayes has 94% accuracy," ask — *accuracy at what cost?*
- **Week 08** (SVM): Before choosing an SVM kernel, check — *am I overfitting to the training set?*

> **Key Insight: Model Evaluation is Not a Step — It's a Mindset**
>
> Evaluation pervades the **entire** workflow. From data collection (is this representative?) to deployment (is the model still accurate on new data?).

### Machine Learning in Civil Engineering

Today we explored ML across five CE sub-disciplines:

1. **Energy & Sustainability** — predicting building heating loads from geometry
2. **Geotechnical Engineering** — soil liquefaction screening from borehole data
3. **Transportation Engineering** — modeling traffic speed-density relationships
4. **Structural/Earthquake Engineering** — detecting data leakage in damage prediction
5. **Water Infrastructure** — prioritizing pipe inspections with limited budgets

> **The Right Question**
>
> Don't ask: *"What's my model's accuracy?"*
>
> Ask: *"What's the cost of my model's errors, and can I live with it?"*
>
> This question should follow you into every ML project you work on — in this course and beyond.

---

### Questions?

**Dr. Eyuphan Koc**
eyuphan.koc@bogazici.edu.tr